In [1]:
# IMPORTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [3]:
file_path = "dataset_ovarian_cancer.txt"

df = pd.read_csv(
    file_path,
    sep="\t",
    skiprows=5
)

print(df.head())
print()
print("Shape:", df.shape)
print()
print("Columns:")
print(df.columns)

  Sample-ID  Replicate     PROK1       WFDC2        KRT19    FR-alpha  \
0      id_1          1  51.27954  2684.67462  814977.3242   978.62687   
1      id_1          2  61.33529  2837.46526  896027.9581  1105.94503   
2      id_2          1  24.85198  2330.02736  791045.9677   839.24954   
3      id_2          2  40.97260  2827.01105  1003158.814  1037.14645   
4      id_3          1  26.02276  2398.98013  1193382.307  1045.50094   

       ICOSLG         CDH3        MSMB    CLEC6A    TACSTD2       MUC-16  \
0  4962.66112  38555.25779  4923.13477  15.18093  160.80636  16652.85209   
1  5416.43019  39359.37085  4895.63176  14.14949  161.81711  17377.84122   
2  5625.48669  30198.26938  4453.04688  12.05641  130.42806   30700.9644   
3  7302.93771  34436.35092  5443.09544  15.06589  153.65555  43507.23046   
4  5224.95801  20999.56857  5890.22654   7.07682  172.29628  28569.18683   

       SPINT1  AgeAtDiag Diagnosis  
0  9610.23806         49         I  
1   8858.6119         49      

In [4]:
features = [
    "PROK1",
    "WFDC2",
    "KRT19",
    "FR-alpha",
    "ICOSLG",
    "CDH3",
    "MSMB",
    "CLEC6A",
    "TACSTD2",
    "MUC-16",
    "SPINT1",
    "AgeAtDiag"
]

target = "Diagnosis"

uloq = {
    "PROK1": 16883.06305,
    "WFDC2": 181210.6285,
    "KRT19": 68900244.48,
    "FR-alpha": 163421.4035,
    "ICOSLG": 55648.31473,
    "CDH3": 638185.7656,
    "MSMB": 177291.4083,
    "CLEC6A": 13291.04962,
    "TACSTD2": 33922.31855,
    "MUC-16": 2288168.669,
    "SPINT1": 193069.9522
}

for feature in uloq:
    df[feature] = df[feature].replace(
        "> ULOQ",
        uloq[feature]
    )

    df[feature] = pd.to_numeric(
        df[feature],
        errors="coerce"
    )

print("Missing values:")
print(df[features].isna().sum())

Missing values:
PROK1        7
WFDC2        7
KRT19        7
FR-alpha     7
ICOSLG       7
CDH3         7
MSMB         7
CLEC6A       7
TACSTD2      7
MUC-16       7
SPINT1       7
AgeAtDiag    0
dtype: int64


In [5]:
df_patient = (
    df.groupby("Sample-ID")
      .agg({
          **{feature: "mean" for feature in features},
          "Diagnosis": "first"
      })
      .reset_index()
)

print("Patient-level dataset:")
print(df_patient.head())

print()
print("Shape:", df_patient.shape)

Patient-level dataset:
  Sample-ID       PROK1        WFDC2         KRT19     FR-alpha       ICOSLG  \
0      id_1   56.307415  2761.069940  8.555026e+05  1042.285950  5189.545655   
1     id_10   60.051290  2692.821970  7.599913e+05   911.213855  5974.368245   
2    id_100   70.799593  2208.716773  6.956947e+05   866.758660  6208.133080   
3    id_101   55.791713  2176.080353  1.006915e+06   829.835973  4927.174610   
4    id_102  158.137193  3138.790933  9.798309e+05   891.867067  4373.818200   

            CDH3         MSMB     CLEC6A     TACSTD2         MUC-16  \
0   38957.314320  4909.383265  14.665210  161.311735   17015.346655   
1   28535.242535  1702.930035   9.942800  192.346485    3963.833860   
2   43719.793630  3900.999347  14.356180  189.454073  109807.330633   
3   28681.659173  3558.996607  13.050203  101.552563   15534.116423   
4  184746.854733  3592.575700   7.482760  158.556143   54909.897717   

         SPINT1  AgeAtDiag   Diagnosis  
0   9234.424980       49.0  

In [7]:
X = df_patient[features].copy()
y = df_patient[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:")
print(label_encoder.classes_)

X shape: (224, 12)
y shape: (224,)
Classes:
['I' 'II' 'III' 'IV' 'benign' 'borderline']


In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (156, 12)
Validation: (34, 12)
Test: (34, 12)


In [9]:
imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)

X_val = imputer.transform(X_val)

X_test = imputer.transform(X_test)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)

X_test = scaler.transform(X_test)

In [10]:
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

In [11]:
batch_size = 16

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [12]:
print("Train:", X_train_tensor.shape)
print("Validation:", X_val_tensor.shape)
print("Test:", X_test_tensor.shape)

print()

print("Example X:")
print(X_train_tensor[0])

print()

print("Example y:")
print(y_train_tensor[0])

print()

print("Classes:")
print(label_encoder.classes_)

Train: torch.Size([156, 12])
Validation: torch.Size([34, 12])
Test: torch.Size([34, 12])

Example X:
tensor([ 7.3616, -0.5484, -0.3669, -0.2711, -0.3894, -0.5102, -0.4278, -0.7111,
        -0.4575, -0.5569, -0.1329, -1.4606])

Example y:
tensor(5)

Classes:
['I' 'II' 'III' 'IV' 'benign' 'borderline']
